In [1]:
# Cell 0: Mount Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

#!pip install torch torchvision torchaudio
!pip install matplotlib numpy pillow scikit-image opencv-python pandas seaborn imageio
!pip install -q imageio-ffmpeg

Mounted at /content/drive


In [2]:
# Cell 0b: Clone FastDVDNet repo
!git clone https://github.com/m-tassano/fastdvdnet /content/fastdvdnet
print("✓ FastDVDNet repo cloned")

Cloning into '/content/fastdvdnet'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 145 (delta 21), reused 12 (delta 9), pack-reused 109 (from 1)
Receiving objects: 100% (145/145), 34.97 MiB | 46.75 MiB/s, done.
Resolving deltas: 100% (71/71), done.
✓ FastDVDNet repo cloned


In [3]:
# Cell 1: Imports and device configuration
import sys
sys.path.append('/content/drive/MyDrive/ResearchProject')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import time

from unet_denoiser_v2 import BlindVideoDenoiserUNet
from video_io import VideoLoader, VideoSaver, VideoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

project_dir = Path('/content/drive/MyDrive/ResearchProject')
benchmark_dir = project_dir / 'benchmark_results_demosaicing'
benchmark_dir.mkdir(exist_ok=True)

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.4 GB


In [4]:
# Cell 2: Import the framework modules
# (If using files on Drive, just import them. If pasting inline, use the
#  inverse_problem_framework.py contents from the file I gave you.)

from inverse_problem_framework import LinearOperator, VideoDenoiser, KadkhodaieSolver
from linear_operators import DemosaicingOperator, create_bayer_mosaic
from denoiser_wrappers import BlindVideoDenoiserWrapper
from inverse_framework_benchmark import DemosaicingBenchmark

print("✓ Framework imported")

✓ Framework imported


In [5]:
# Cell 3: No separate operator definition needed.
# The DemosaicingOperator is created per-video (since it depends on H, W).
# We'll create it in the test cells below.
print("✓ DemosaicingOperator will be created per-video (depends on H, W)")

✓ DemosaicingOperator will be created per-video (depends on H, W)


In [6]:
class BlindVideoDenoiserWrapper(VideoDenoiser):
    def __init__(self, model, device='cuda', pad_to=32, num_input_frames=5):
        self.model = model.to(device)
        self.device = device
        self.pad_to = pad_to
        self.num_input_frames = num_input_frames
        self.half_window = num_input_frames // 2
        self.model.eval()

    def _pad(self, frame):
        C, H, W = frame.shape
        ph = (self.pad_to - H % self.pad_to) % self.pad_to
        pw = (self.pad_to - W % self.pad_to) % self.pad_to
        if ph > 0 or pw > 0:
            frame = F.pad(frame, (0, pw, 0, ph), mode='reflect')
        return frame, (H, W)

    def _unpad(self, frame, orig):
        return frame[:, :orig[0], :orig[1]]

    def denoise(self, noisy_video, noise_std=0.0):
        T, C, H, W = noisy_video.shape
        noisy_video = noisy_video.to(self.device)
        frames = []
        with torch.no_grad():
            for t in range(T):
                indices = [max(0, min(T-1, t+off)) for off in range(-self.half_window, self.half_window+1)]
                neighbor_frames = [noisy_video[i] for i in indices]

                padded = []
                orig_size = None
                for f in neighbor_frames:
                    pf, orig_size = self._pad(f)
                    padded.append(pf)

                concat = torch.cat(padded, dim=0).unsqueeze(0)
                out = self.model(concat)
                denoised = self._unpad(out.squeeze(0), orig_size)
                frames.append(torch.clamp(denoised, 0, 1))
        return torch.stack(frames)

    def denoise_frame(self, prev_frame, curr_frame, next_frame, noise_std=0.0):
        video = torch.stack([prev_frame, prev_frame, curr_frame, next_frame, next_frame])
        return self.denoise(video, noise_std)[2]

print("✓ Denoiser wrapper defined (5-frame, pad_to=32)")

✓ Denoiser wrapper defined (5-frame, pad_to=32)


In [8]:
# Cell 5: Load your trained denoiser
print("Loading trained blind video denoiser...")

checkpoint_path = project_dir / 'checkpoints_large_model' / 'best_model_final.pt'

model = BlindVideoDenoiserUNet(num_input_frames=5, out_channels=3, base_channels=64)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Loaded from epoch {checkpoint['epoch']}")
print(f"  Train loss: {checkpoint['train_loss']:.6f}")
print(f"  Val loss:   {checkpoint['val_loss']:.6f}")

# Wrap for framework — pad_to=8 for 3-stage UNet (2^3=8)
your_denoiser = BlindVideoDenoiserWrapper(model, device=device, pad_to=8)
print("✓ Denoiser wrapped and ready")

Loading trained blind video denoiser...
✓ Loaded from epoch 26
  Train loss: 19.711076
  Val loss:   19.254098
✓ Denoiser wrapped and ready


In [9]:
# Cell 5b: Load FastDVDNet and wrap for the solver
import sys
import os
fastdvdnet_repo = '/content/fastdvdnet'
if fastdvdnet_repo not in sys.path:
    sys.path.insert(0, fastdvdnet_repo)

from models import FastDVDnet as OfficialFastDVDnet

# Load pretrained FastDVDNet
fdvd_model = OfficialFastDVDnet(num_input_frames=5)
fdvd_state = torch.load(os.path.join(fastdvdnet_repo, 'model.pth'), map_location=device)
if any(k.startswith('module.') for k in fdvd_state.keys()):
    fdvd_state = {k.replace('module.', ''): v for k, v in fdvd_state.items()}
fdvd_model.load_state_dict(fdvd_state)
fdvd_model = fdvd_model.to(device).eval()
print(f"✓ FastDVDNet loaded ({sum(p.numel() for p in fdvd_model.parameters()):,} params)")

# FastDVDNet wrapper for the Kadkhodaie solver
class FastDVDNetSolverWrapper(VideoDenoiser):
    """
    Wraps FastDVDNet for the inverse problem solver.
    Uses the official denoise_seq_fastdvdnet function which handles
    the two-stage architecture and 5-frame windowing correctly.
    """

    def __init__(self, model, repo_path, device='cuda', pad_to=4):
        self.model = model.to(device).eval()
        self.device = device
        self.pad_to = pad_to  # FastDVDNet needs dimensions divisible by 4

        # Import their official denoising function
        import sys
        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)
        from fastdvdnet import denoise_seq_fastdvdnet
        self.denoise_seq_fn = denoise_seq_fastdvdnet

    def denoise(self, noisy_video: torch.Tensor, noise_std: float = 0.0) -> torch.Tensor:
        T, C, H, W = noisy_video.shape
        noisy_video = noisy_video.to(self.device)

        # Pad to be divisible by 4 if needed
        pad_h = (self.pad_to - H % self.pad_to) % self.pad_to
        pad_w = (self.pad_to - W % self.pad_to) % self.pad_to
        if pad_h > 0 or pad_w > 0:
            noisy_video = F.pad(noisy_video, (0, pad_w, 0, pad_h), mode='reflect')

        # Noise sigma as tensor (0-1 range)
        sigma_tensor = torch.FloatTensor([noise_std]).to(self.device)

        with torch.no_grad():
            denoised = self.denoise_seq_fn(
                seq=noisy_video,
                noise_std=sigma_tensor,
                temp_psz=5,
                model_temporal=self.model
            )

        # Remove padding
        if pad_h > 0 or pad_w > 0:
            denoised = denoised[:, :, :H, :W]

        return torch.clamp(denoised, 0, 1)

    def denoise_frame(self, prev_frame, curr_frame, next_frame, noise_std=0.0):
        video = torch.stack([prev_frame, prev_frame, curr_frame, next_frame, next_frame])
        return self.denoise(video, noise_std)[2]


fastdvdnet_denoiser = FastDVDNetSolverWrapper(
    fdvd_model, repo_path=fastdvdnet_repo, device=device, pad_to=4
)
print("✓ FastDVDNet wrapped for solver")

✓ FastDVDNet loaded (2,479,096 params)
✓ FastDVDNet wrapped for solver


In [12]:
# Cell 7: Load test videos for benchmark
print("Loading test videos...")
davis_root = Path('/content/drive/MyDrive/ResearchProject/DAVISDataset')
test_videos = {}
video_dirs = sorted([d for d in davis_root.iterdir() if d.is_dir()])[:5]

for vdir in video_dirs:
    name = vdir.name
    print(f"  Loading {name}...", end=' ', flush=True)
    try:
        v = VideoLoader.load_frame_sequence(
            str(vdir), max_frames=None, resize=(512, 832), device=device
        )
        test_videos[name] = v
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nLoaded {len(test_videos)} test videos")

Loading test videos...
  Loading baseball... Loaded 90 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/baseball
  Shape: torch.Size([90, 3, 512, 832]) (T, C, H, W)
✓
  Loading basketball-game... Loaded 77 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/basketball-game
  Shape: torch.Size([77, 3, 512, 832]) (T, C, H, W)
✓
  Loading bear... Loaded 82 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bear
  Shape: torch.Size([82, 3, 512, 832]) (T, C, H, W)
✓
  Loading bears-ball... Loaded 78 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bears-ball
  Shape: torch.Size([78, 3, 512, 832]) (T, C, H, W)
✓
  Loading bike-packing... Loaded 69 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bike-packing
  Shape: torch.Size([69, 3, 512, 832]) (T, C, H, W)
✓

Loaded 5 test videos


In [13]:
# Cell 7: DEMOSAICING — Run solver with BOTH denoisers on all test videos
print("=" * 60)
print("DEMOSAICING TEST (RGGB Bayer) — YourUNet + FastDVDNet")
print("=" * 60)

from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

all_demosaic_results = {}

denoisers = {
    'YourUNet': your_denoiser,
    'FastDVDNet': fastdvdnet_denoiser,
}

for video_name, clean in test_videos.items():
    T, C, H, W = clean.shape
    print(f"\n{'─'*50}")
    print(f"{video_name} ({T} frames, {W}×{H})")
    print(f"{'─'*50}")

    operator, mosaiced, naive_demosaiced = create_bayer_mosaic(
        clean, H, W, pattern='RGGB', device=device
    )

    video_results = {
        'clean': clean,
        'mosaiced': mosaiced,
        'naive': naive_demosaiced,
    }

    for den_name, denoiser in denoisers.items():
        print(f"\n  [{den_name}] Solving...", flush=True)
        solver = KadkhodaieSolver(operator, denoiser, device=device)

        restored, metrics = solver.solve(
            mosaiced,
            sigma_0=0.1,
            sigma_L=0.002,
            h0=0.01,
            beta=0.1,
            max_iterations=2000,
            verbose=True,
            log_freq=200,
        )

        video_results[f'restored_{den_name}'] = restored
        video_results[f'metrics_{den_name}'] = metrics

        # Quick summary
        mid = T // 2
        c = clean[mid].permute(1, 2, 0).cpu().numpy()
        r = restored[mid].permute(1, 2, 0).cpu().numpy()
        print(f"  [{den_name}] PSNR: {compute_psnr(c, np.clip(r,0,1), data_range=1.0):.2f} dB")

    all_demosaic_results[video_name] = video_results

print(f"\n{'=' * 60}")
print(f"Done — {len(all_demosaic_results)} videos × {len(denoisers)} denoisers")
print(f"{'=' * 60}")

DEMOSAICING TEST (RGGB Bayer) — YourUNet + FastDVDNet

──────────────────────────────────────────────────
baseball (90 frames, 832×512)
──────────────────────────────────────────────────

  [YourUNet] Solving...
  Converged in 92 iters (144.6s), final sigma=0.001990
  [YourUNet] PSNR: 34.53 dB

  [FastDVDNet] Solving...
  Converged in 108 iters (296.4s), final sigma=0.001970
  [FastDVDNet] PSNR: 32.33 dB

──────────────────────────────────────────────────
basketball-game (77 frames, 832×512)
──────────────────────────────────────────────────

  [YourUNet] Solving...
  Converged in 95 iters (127.8s), final sigma=0.001856
  [YourUNet] PSNR: 39.11 dB

  [FastDVDNet] Solving...
  Converged in 100 iters (234.7s), final sigma=0.001974
  [FastDVDNet] PSNR: 37.83 dB

──────────────────────────────────────────────────
bear (82 frames, 832×512)
──────────────────────────────────────────────────

  [YourUNet] Solving...
  Converged in 89 iters (127.5s), final sigma=0.001954
  [YourUNet] PSNR: 37.

In [14]:
# Cell 8: Visualize — one frame at a time, all denoisers
frames_per_video = 6
denoiser_names = ['YourUNet', 'FastDVDNet']

for video_name, data in all_demosaic_results.items():
    clean = data['clean']
    mosaiced = data['mosaiced']
    naive = data['naive']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    print(f"\n--- {video_name} ({T} frames) ---")

    for idx in frame_indices:
        c_np = clean[idx].permute(1, 2, 0).cpu().numpy()
        m_np = mosaiced[idx].permute(1, 2, 0).cpu().numpy()
        n_np = naive[idx].permute(1, 2, 0).cpu().numpy()

        # 2 rows: top = Clean, Mosaiced | bottom = one per denoiser
        ncols = max(2, len(denoiser_names))
        fig, axes = plt.subplots(2, ncols, figsize=(8 * ncols, 12))

        # Top-left: Clean
        axes[0, 0].imshow(np.clip(c_np, 0, 1))
        axes[0, 0].set_title('Clean (Ground Truth)', fontsize=13, fontweight='bold')
        axes[0, 0].axis('off')

        # Top-right: Mosaiced
        p_m = compute_psnr(c_np, np.clip(m_np, 0, 1), data_range=1.0)
        axes[0, 1].imshow(np.clip(m_np, 0, 1))
        axes[0, 1].set_title(f'Mosaiced — PSNR={p_m:.1f} dB', fontsize=13, fontweight='bold')
        axes[0, 1].axis('off')

        # Hide extra top row cells if more than 2 denoisers
        for j in range(2, ncols):
            axes[0, j].axis('off')

        # Bottom row: one per denoiser
        for j, den_name in enumerate(denoiser_names):
            r_np = data[f'restored_{den_name}'][idx].permute(1, 2, 0).cpu().numpy()
            p_r = compute_psnr(c_np, np.clip(r_np, 0, 1), data_range=1.0)
            axes[1, j].imshow(np.clip(r_np, 0, 1))
            axes[1, j].set_title(f'{den_name} — PSNR={p_r:.1f} dB',
                                 fontsize=13, fontweight='bold')
            axes[1, j].axis('off')

        # Hide extra bottom cells
        for j in range(len(denoiser_names), ncols):
            axes[1, j].axis('off')

        plt.suptitle(f'{video_name} — Frame {idx+1}/{T}',
                     fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [15]:
import subprocess

output_dir = '/content/drive/MyDrive/ResearchProject/demosaicing_videos_architecture2'
os.makedirs(output_dir, exist_ok=True)
fps = 24

for video_name, data in all_demosaic_results.items():
    clean = data['clean']
    mosaiced = data['mosaiced']
    T = clean.shape[0]

    videos_to_save = {
        'clean': clean,
        'mosaiced': mosaiced,
        'naive_interp': data['naive'],
        'solver_YourUNet': data['restored_YourUNet'],
        'solver_FastDVDNet': data['restored_FastDVDNet'],
    }

    for label, tensor in videos_to_save.items():
        frames_np = [tensor[t].permute(1, 2, 0).cpu().numpy() for t in range(T)]
        h, w = frames_np[0].shape[:2]
        save_path = os.path.join(output_dir, f"{video_name}_{label}.mp4")

        cmd = [
            'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
            '-pix_fmt', 'rgb24', '-s', f'{w}x{h}', '-r', str(fps),
            '-i', '-', '-c:v', 'libx264', '-crf', '0',
            '-preset', 'medium', '-pix_fmt', 'yuv444p', save_path
        ]
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE,
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for frame in frames_np:
            proc.stdin.write((np.clip(frame, 0, 1) * 255).astype(np.uint8).tobytes())
        proc.stdin.close()
        proc.wait()

    print(f"Saved {len(videos_to_save)} videos for {video_name}")

print(f"\nAll videos saved to: {output_dir}")

Saved 5 videos for baseball
Saved 5 videos for basketball-game
Saved 5 videos for bear
Saved 5 videos for bears-ball
Saved 5 videos for bike-packing

All videos saved to: /content/drive/MyDrive/ResearchProject/demosaicing_videos_architecture2


In [16]:
# Cell 8b: Save 6 frames per video as PNG images
import os

frames_per_video = 6
denoiser_names = ['YourUNet', 'FastDVDNet']
save_root = str(benchmark_dir / 'demosaicing_frames')

for video_name, data in all_demosaic_results.items():
    clean = data['clean']
    mosaiced = data['mosaiced']
    naive = data['naive']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    video_dir = os.path.join(save_root, video_name)
    os.makedirs(video_dir, exist_ok=True)

    for idx in frame_indices:
        # Save each version as PNG
        versions = {
            'clean': clean[idx],
            'mosaiced': mosaiced[idx],
            'naive_interp': naive[idx],
        }
        for den_name in denoiser_names:
            versions[f'solver_{den_name}'] = data[f'restored_{den_name}'][idx]

        for label, tensor in versions.items():
            img_np = tensor.permute(1, 2, 0).cpu().numpy()
            img_uint8 = (np.clip(img_np, 0, 1) * 255).astype(np.uint8)
            from PIL import Image
            img_pil = Image.fromarray(img_uint8)
            save_path = os.path.join(video_dir, f'frame{idx+1:03d}_{label}.png')
            img_pil.save(save_path)

    print(f"Saved {len(frame_indices)} frames × {len(versions)} versions for {video_name}")

print(f"\nAll frames saved to: {save_root}")

Saved 6 frames × 5 versions for baseball
Saved 6 frames × 5 versions for basketball-game
Saved 6 frames × 5 versions for bear
Saved 6 frames × 5 versions for bears-ball
Saved 6 frames × 5 versions for bike-packing

All frames saved to: /content/drive/MyDrive/ResearchProject/benchmark_results_demosaicing/demosaicing_frames


In [17]:
# Cell 12: Generate Excel with Naive + YourUNet + FastDVDNet results
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from datetime import datetime

denoiser_names = ['YourUNet', 'FastDVDNet']

# Compute per-video averages
video_summary = []
for video_name, data in all_demosaic_results.items():
    clean = data['clean']
    naive = data['naive']
    T = clean.shape[0]

    row = {'video': video_name, 'frames': T}

    # Naive baseline
    psnr_n, ssim_n = [], []
    for t in range(T):
        c = clean[t].permute(1, 2, 0).cpu().numpy()
        n = naive[t].permute(1, 2, 0).cpu().numpy()
        psnr_n.append(compute_psnr(c, np.clip(n, 0, 1), data_range=1.0))
        ssim_n.append(compute_ssim(c, np.clip(n, 0, 1), data_range=1.0, channel_axis=2))
    row['psnr_naive'] = np.mean(psnr_n)
    row['ssim_naive'] = np.mean(ssim_n)

    # Each denoiser
    for den_name in denoiser_names:
        psnr_d, ssim_d = [], []
        restored = data[f'restored_{den_name}']
        for t in range(T):
            c = clean[t].permute(1, 2, 0).cpu().numpy()
            r = restored[t].permute(1, 2, 0).cpu().numpy()
            psnr_d.append(compute_psnr(c, np.clip(r, 0, 1), data_range=1.0))
            ssim_d.append(compute_ssim(c, np.clip(r, 0, 1), data_range=1.0, channel_axis=2))
        row[f'psnr_{den_name}'] = np.mean(psnr_d)
        row[f'ssim_{den_name}'] = np.mean(ssim_d)

    video_summary.append(row)

# --- Create Excel ---
wb = openpyxl.Workbook()
header_font = Font(bold=True, size=12)
title_font = Font(bold=True, size=14)
header_fill = PatternFill(start_color='D5E8F0', end_color='D5E8F0', fill_type='solid')
best_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_cell(ws, row, col, fmt=None, bold=False, fill=None):
    cell = ws.cell(row=row, column=col)
    cell.border = thin_border
    cell.alignment = Alignment(horizontal='center')
    if fmt: cell.number_format = fmt
    if bold: cell.font = header_font
    if fill: cell.fill = fill

# ===== Sheet 1: PSNR =====
ws1 = wb.active
ws1.title = 'PSNR Comparison'
ws1.cell(row=1, column=1, value='Demosaicing PSNR Comparison (dB)').font = title_font

headers = ['Video', 'Frames', 'Naive Interp'] + denoiser_names + ['Best']
for col, h in enumerate(headers, 1):
    ws1.cell(row=3, column=col, value=h)
    ws1.cell(row=3, column=col).font = header_font
    ws1.cell(row=3, column=col).fill = header_fill
    ws1.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    psnr_vals = {'Naive Interp': v['psnr_naive']}
    psnr_vals.update({dn: v[f'psnr_{dn}'] for dn in denoiser_names})
    best_name = max(psnr_vals, key=psnr_vals.get)

    ws1.cell(row=row, column=1, value=v['video'])
    ws1.cell(row=row, column=2, value=v['frames'])
    ws1.cell(row=row, column=3, value=round(v['psnr_naive'], 2))
    fill = best_fill if best_name == 'Naive Interp' else None
    style_cell(ws1, row, 3, '0.00', fill=fill)
    for j, dn in enumerate(denoiser_names):
        col = 4 + j
        ws1.cell(row=row, column=col, value=round(v[f'psnr_{dn}'], 2))
        fill = best_fill if dn == best_name else None
        style_cell(ws1, row, col, '0.00', fill=fill)
    ws1.cell(row=row, column=4 + len(denoiser_names), value=round(psnr_vals[best_name], 2))
    for col in [1, 2, 4 + len(denoiser_names)]:
        style_cell(ws1, row, col, '0.00' if col >= 3 else None)

# Average row
ar = 4 + len(video_summary)
ws1.cell(row=ar, column=1, value='AVERAGE').font = header_font
ws1.cell(row=ar, column=3, value=round(np.mean([v['psnr_naive'] for v in video_summary]), 2))
for j, dn in enumerate(denoiser_names):
    ws1.cell(row=ar, column=4+j, value=round(np.mean([v[f'psnr_{dn}'] for v in video_summary]), 2))
for col in range(1, len(headers)+1):
    style_cell(ws1, ar, col, '0.00' if col >= 3 else None, bold=True)

ws1.column_dimensions['A'].width = 22
for c in 'BCDEFGH':
    ws1.column_dimensions[c].width = 15

# ===== Sheet 2: SSIM =====
ws2 = wb.create_sheet('SSIM Comparison')
ws2.cell(row=1, column=1, value='Demosaicing SSIM Comparison').font = title_font

for col, h in enumerate(headers, 1):
    ws2.cell(row=3, column=col, value=h)
    ws2.cell(row=3, column=col).font = header_font
    ws2.cell(row=3, column=col).fill = header_fill
    ws2.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    ssim_vals = {'Naive Interp': v['ssim_naive']}
    ssim_vals.update({dn: v[f'ssim_{dn}'] for dn in denoiser_names})
    best_name = max(ssim_vals, key=ssim_vals.get)

    ws2.cell(row=row, column=1, value=v['video'])
    ws2.cell(row=row, column=2, value=v['frames'])
    ws2.cell(row=row, column=3, value=round(v['ssim_naive'], 4))
    fill = best_fill if best_name == 'Naive Interp' else None
    style_cell(ws2, row, 3, '0.0000', fill=fill)
    for j, dn in enumerate(denoiser_names):
        col = 4 + j
        ws2.cell(row=row, column=col, value=round(v[f'ssim_{dn}'], 4))
        fill = best_fill if dn == best_name else None
        style_cell(ws2, row, col, '0.0000', fill=fill)
    ws2.cell(row=row, column=4 + len(denoiser_names), value=round(ssim_vals[best_name], 4))
    for col in [1, 2, 4 + len(denoiser_names)]:
        style_cell(ws2, row, col, '0.0000' if col >= 3 else None)

ar2 = 4 + len(video_summary)
ws2.cell(row=ar2, column=1, value='AVERAGE').font = header_font
ws2.cell(row=ar2, column=3, value=round(np.mean([v['ssim_naive'] for v in video_summary]), 4))
for j, dn in enumerate(denoiser_names):
    ws2.cell(row=ar2, column=4+j, value=round(np.mean([v[f'ssim_{dn}'] for v in video_summary]), 4))
for col in range(1, len(headers)+1):
    style_cell(ws2, ar2, col, '0.0000' if col >= 3 else None, bold=True)

ws2.column_dimensions['A'].width = 22
for c in 'BCDEFGH':
    ws2.column_dimensions[c].width = 15

# ===== Sheet 3: PSNR Gain =====
ws3 = wb.create_sheet('PSNR Gain')
ws3.cell(row=1, column=1, value='PSNR Improvement over Naive Interpolation (dB)').font = title_font

h3 = ['Video', 'Naive Interp'] + denoiser_names
for col, h in enumerate(h3, 1):
    ws3.cell(row=3, column=col, value=h)
    ws3.cell(row=3, column=col).font = header_font
    ws3.cell(row=3, column=col).fill = header_fill
    ws3.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    ws3.cell(row=row, column=1, value=v['video'])
    ws3.cell(row=row, column=2, value=0.00)  # Naive vs itself = 0
    style_cell(ws3, row, 2, '0.00')
    gains = {}
    for j, dn in enumerate(denoiser_names):
        gain = v[f'psnr_{dn}'] - v['psnr_naive']
        gains[dn] = gain
        ws3.cell(row=row, column=3+j, value=round(gain, 2))
    best_name = max(gains, key=gains.get)
    for j, dn in enumerate(denoiser_names):
        fill = best_fill if dn == best_name else None
        style_cell(ws3, row, 3+j, '0.00', fill=fill)
    style_cell(ws3, row, 1)

ar3 = 4 + len(video_summary)
ws3.cell(row=ar3, column=1, value='AVERAGE').font = header_font
ws3.cell(row=ar3, column=2, value=0.00)
style_cell(ws3, ar3, 2, '0.00', bold=True)
for j, dn in enumerate(denoiser_names):
    avg_gain = np.mean([v[f'psnr_{dn}'] - v['psnr_naive'] for v in video_summary])
    ws3.cell(row=ar3, column=3+j, value=round(avg_gain, 2))
    style_cell(ws3, ar3, 3+j, '0.00', bold=True)
style_cell(ws3, ar3, 1, bold=True)

ws3.column_dimensions['A'].width = 22
for c in 'BCDE':
    ws3.column_dimensions[c].width = 15

# ===== Sheet 4: Notes =====
ws4 = wb.create_sheet('Notes')
ws4.cell(row=1, column=1, value='Demosaicing Benchmark — Methodology Notes').font = title_font
notes = [
    ('Date:', str(datetime.now())),
    ('Test videos:', str(len(video_summary))),
    ('Processing resolution:', f'{clean.shape[3]}×{clean.shape[2]}'),
    ('Bayer pattern:', 'RGGB'),
    ('Solver:', 'Kadkhodaie & Simoncelli (2021)'),
    ('sigma_0:', '0.1'), ('sigma_L:', '0.002'), ('h0:', '0.01'), ('beta:', '0.1'),
    ('', ''),
    ('Metrics:', ''),
    ('PSNR', 'Peak Signal-to-Noise Ratio vs clean ground truth (dB). Higher = better.'),
    ('SSIM', 'Structural Similarity vs clean ground truth (0 to 1). Higher = better.'),
    ('PSNR Gain', 'Improvement over naive bilinear interpolation baseline (dB).'),
    ('', ''),
    ('Model Notes:', ''),
    ('YourUNet', 'Blind bias-free UNet. Trained on DAVIS, σ ∈ [5, 250]. 3 stages, 64 base ch.'),
    ('FastDVDNet', 'Pretrained from official repo. Trained on σ ∈ [5, 55]. Uses noise sigma as input.'),
    ('Naive Interp', 'Simple 3×3 averaging of observed CFA values per channel. Baseline.'),
    ('', ''),
    ('Green cells', 'indicate the best performer for each video (including naive).'),
]
for i, (k, val) in enumerate(notes):
    ws4.cell(row=3+i, column=1, value=k).font = Font(bold=True) if k else Font()
    ws4.cell(row=3+i, column=2, value=val)
ws4.column_dimensions['A'].width = 22
ws4.column_dimensions['B'].width = 75

excel_path = str(benchmark_dir / 'demosaicing_benchmark_comparison.xlsx')
wb.save(excel_path)
print(f"✓ Excel saved: {excel_path}")
print(f"  Sheets: {wb.sheetnames}")
print(f"  Green cells = best performer per video (including naive)")

✓ Excel saved: /content/drive/MyDrive/ResearchProject/benchmark_results_demosaicing/demosaicing_benchmark_comparison.xlsx
  Sheets: ['PSNR Comparison', 'SSIM Comparison', 'PSNR Gain', 'Notes']
  Green cells = best performer per video (including naive)
